# 3 · ResNet18 fine-tuning
**Owner:** Amrit · **Phase 1:** GPU smoke test · **Phase 3:** training

On Colab: **Runtime → Change runtime type → T4 GPU**.

In [1]:
# ---- Setup: works on Colab and locally. Run this first. ----
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Shared Drive folder (add a shortcut to it in your "My Drive" so this path exists)
    DRIVE_DIR = Path("/content/drive/MyDrive/image-processing-group18")
    REPO_DIR = Path("/content/image-processing-group18")
    if not REPO_DIR.exists():
        # If the repo is private, use https://<YOUR_GITHUB_TOKEN>@github.com/... here
        subprocess.run(["git", "clone", "https://github.com/aylat7/image-processing-group18.git", str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # Copy the raw dataset from Drive to Colab's fast local disk (once per session)
    if not Path("data/raw/asl_raw").exists():
        os.makedirs("data/raw", exist_ok=True)
        subprocess.run(["unzip", "-q", str(DRIVE_DIR / "asl_raw.zip"), "-d", "data/raw"], check=True)
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
import common
print("Project root:", os.getcwd(), "| Colab:", IN_COLAB)

Project root: /home/roneet/Desktop/IMG467-project | Colab: False


## Phase 1 · GPU smoke test
Check that a pretrained ResNet18 loads, gets a 26-class head, and can train for a few steps on real images.

In [2]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (fine locally, use GPU on Colab)")

common.set_seed()
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, common.NUM_CLASSES)   # 512 -> 26
model = model.to(device)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

torch 2.15.0+cu132 | device: cpu | no GPU (fine locally, use GPU on Colab)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/roneet/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


0.3%

0.6%

0.8%

1.1%

1.4%

... (350 outputs left out)

99.6%

99.9%

100.0%

Trainable params: 11189850


In [3]:
# A small batch of real images, run through the SAME preprocessing as the webcam app
paths, labels = [], []
for letter in common.CLASSES:
    for f in sorted((common.RAW_DIR / letter).glob("*.jpg"))[:2]:
        paths.append(f); labels.append(common.CLASS_TO_IDX[letter])
x = torch.from_numpy(np.stack([common.preprocess_cnn(common.load_image(p)) for p in paths])).to(device)
y = torch.tensor(labels).to(device)
print("batch:", tuple(x.shape), "labels:", tuple(y.shape))

opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()
model.train()
for step in range(10):
    t = time.time()
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    opt.step()
    print(f"step {step}: loss {loss.item():.3f}  ({(time.time() - t) * 1000:.0f} ms)")
print("Loss should go down -> the training loop works.")

batch: (52, 3, 224, 224) labels: (52,)


step 0: loss 3.404  (2463 ms)


step 1: loss 2.284  (2407 ms)


step 2: loss 1.499  (2388 ms)


step 3: loss 0.974  (2390 ms)


step 4: loss 0.642  (2349 ms)


step 5: loss 0.431  (2367 ms)


step 6: loss 0.295  (2387 ms)


step 7: loss 0.207  (2661 ms)


step 8: loss 0.151  (3283 ms)


step 9: loss 0.115  (3891 ms)
Loss should go down -> the training loop works.


In [4]:
# Export / reload round-trip (this is how the webcam app will load it on CPU)
import tempfile, os
tmp = os.path.join(tempfile.gettempdir(), "smoke_test.pt")
torch.save(model.state_dict(), tmp)
m2 = resnet18(weights=None); m2.fc = nn.Linear(512, common.NUM_CLASSES)
m2.load_state_dict(torch.load(tmp, map_location="cpu"))
print("Reloaded on CPU OK, file size:", round(os.path.getsize(tmp) / 1e6, 1), "MB")

Reloaded on CPU OK, file size: 44.8 MB


---
## Phase 3 · Training *(next)*
Dataset + augmentation → freeze backbone, train head → unfreeze, fine-tune → early stopping on val macro-F1 → save best to Drive → export `models/resnet18.pt`.